<a href="https://colab.research.google.com/github/mhowlin-web/TP_RAG_ARCA/blob/main/01_descarga_corpus_arca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP RAG y Agentes

## Asistente para consultas sobre trámites de Monotributo en ARCA

En este proyecto voy a construir un sistema RAG para responder preguntas sobre
trámites relacionados con el Monotributo utilizando información proveniente de
fuentes oficiales de ARCA.

El sistema buscará información relevante dentro de un conjunto de documentos
oficiales y utilizará un modelo de lenguaje para generar respuestas basadas
únicamente en los documentos recuperados.

En este primer notebook construyo el corpus documental que utilizaré en las
siguientes etapas del proyecto.

El proceso que realizo en este notebook es:

1. Definir las fuentes oficiales.
2. Descargar automáticamente las páginas.
3. Extraer el contenido textual.
4. Realizar una limpieza básica.
5. Guardar los documentos en formato JSON.

Posteriormente utilizaré este corpus para realizar chunking, generar embeddings,
almacenarlos en Pinecone y construir el sistema RAG.

## Instalación de librerías

En esta celda instalo las librerías que necesito para descargar las páginas web
y extraer su contenido.

`requests` me permite realizar solicitudes HTTP para obtener el contenido de las
páginas.

`BeautifulSoup` me permite analizar el HTML y extraer el texto.

In [1]:
!pip install -q requests beautifulsoup4 lxml

## Importación de librerías

En esta celda importo las librerías que utilizaré durante el notebook.

También importo `datetime` para guardar la fecha y hora en la que descargo cada
fuente. Esto permite conocer cuándo fue obtenido el contenido del corpus.

In [2]:
import requests
import json

from bs4 import BeautifulSoup
from datetime import datetime, timezone

## Definición de las fuentes oficiales

En esta celda defino las páginas oficiales de ARCA que utilizaré como corpus
inicial.

Cada fuente tiene un identificador, un título y una URL.

Guardo esta información de manera estructurada porque posteriormente conservaré
los datos de origen como metadatos de cada documento y de cada chunk.

De esta manera, el sistema RAG podrá mantener la relación entre la información
recuperada y su fuente oficial.

In [3]:
FUENTES_ARCA = [
    {
        "id": "inicio",
        "titulo": "Inicio - Ayuda sobre el Monotributo",
        "url": "https://www.arca.gob.ar/monotributo/ayuda/inicio.asp"
    },
    {
        "id": "clave_fiscal",
        "titulo": "Obtención de Clave Fiscal",
        "url": "https://www.arca.gob.ar/monotributo/ayuda/clave-fiscal.asp"
    },
    {
        "id": "constancias",
        "titulo": "Constancias y credenciales",
        "url": "https://www.arca.gob.ar/monotributo/ayuda/constancias-y-credenciales.asp"
    },
    {
        "id": "facturacion",
        "titulo": "Facturación",
        "url": "https://www.arca.gob.ar/monotributo/ayuda/facturacion.asp"
    },
    {
        "id": "recategorizacion",
        "titulo": "Recategorización",
        "url": "https://ftp.arca.gob.ar/monotributo/ayuda/recategorizacion.asp"
    },
    {
        "id": "baja",
        "titulo": "Baja de monotributo",
        "url": "https://www.arca.gob.ar/monotributo/ayuda/baja.asp"
    },
    {
        "id": "desarrollo_actividad",
        "titulo": "Desarrollo de la actividad",
        "url": "https://www.arca.gob.ar/monotributo/ayuda/desarrollo-de-la-actividad.asp"
    },
    {
        "id": "tutoriales",
        "titulo": "Tutoriales sobre Monotributo",
        "url": "https://www.arca.gob.ar/monotributo/ayuda/tutoriales.asp"
    }
]

## Visualización de las fuentes

En esta celda verifico las fuentes que voy a descargar.

Esto permite comprobar que el corpus inicial contiene las URLs esperadas antes
de comenzar la descarga automática.

In [4]:
print("FUENTES DEL CORPUS INICIAL")
print("=" * 80)

for i, fuente in enumerate(FUENTES_ARCA, start=1):
    print(f"\n{i}. {fuente['titulo']}")
    print(f"   ID: {fuente['id']}")
    print(f"   URL: {fuente['url']}")

FUENTES DEL CORPUS INICIAL

1. Inicio - Ayuda sobre el Monotributo
   ID: inicio
   URL: https://www.arca.gob.ar/monotributo/ayuda/inicio.asp

2. Obtención de Clave Fiscal
   ID: clave_fiscal
   URL: https://www.arca.gob.ar/monotributo/ayuda/clave-fiscal.asp

3. Constancias y credenciales
   ID: constancias
   URL: https://www.arca.gob.ar/monotributo/ayuda/constancias-y-credenciales.asp

4. Facturación
   ID: facturacion
   URL: https://www.arca.gob.ar/monotributo/ayuda/facturacion.asp

5. Recategorización
   ID: recategorizacion
   URL: https://ftp.arca.gob.ar/monotributo/ayuda/recategorizacion.asp

6. Baja de monotributo
   ID: baja
   URL: https://www.arca.gob.ar/monotributo/ayuda/baja.asp

7. Desarrollo de la actividad
   ID: desarrollo_actividad
   URL: https://www.arca.gob.ar/monotributo/ayuda/desarrollo-de-la-actividad.asp

8. Tutoriales sobre Monotributo
   ID: tutoriales
   URL: https://www.arca.gob.ar/monotributo/ayuda/tutoriales.asp


## Función para descargar una página

En esta celda creo una función para descargar el contenido HTML de una URL.

Utilizo un `User-Agent` para identificar la solicitud como proveniente de un
cliente HTTP.

También utilizo `raise_for_status()` para detectar errores HTTP. Por ejemplo,
si una página no existe o el servidor devuelve un error, el programa generará
una excepción en lugar de continuar utilizando contenido inválido.

In [5]:
def descargar_pagina(url):

    headers = {
        "User-Agent": (
            "Mozilla/5.0 "
            "(compatible; RAG-ARCA-TP/1.0)"
        )
    }

    respuesta = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    respuesta.raise_for_status()

    return respuesta.text

## Prueba de descarga

En esta celda pruebo la función descargando una de las fuentes.

Por ahora solamente verifico que puedo obtener contenido HTML desde la página
oficial antes de continuar con la extracción del texto.

In [6]:
html_prueba = descargar_pagina(
    FUENTES_ARCA[0]["url"]
)

print(html_prueba[:1000])

<!DOCTYPE html>
<html lang="es">

    <head>
        <meta charset="utf-8">
        <meta http-equiv="X-UA-Compatible" content="IE=edge">
        <meta name="viewport" content="width=device-width, initial-scale=1">
        <!-- Meta para los Buscadores -->
        <title>Inicio - Ayuda sobre el monotributo - Monotributo | ARCA</title>
        <meta name="description" content="Toda la informaciÃ³n sobre cÃ³mo darte de alta y realizar gestiones como monotributista">
        <meta name="keywords" content="monotributo, impositivo, montos, recategorizaciÃ³n, rÃ©gimen,">
        <meta name="author" content="ARCA">
        <meta name="robots" content="Index, Follow">
        <!-- opciones: Index, Follow - NoIndex, Follow - Index, NoFollow - NoIndex, NoFollow -->
        <!-- Facebook , google + -->
        <meta property="og:title" content="Inicio - Ayuda sobre el monotributo - Monotributo | ARCA">
        <meta property="og:type" content="article">
        <meta property="og


## Extracción del texto

En esta celda creo una función para convertir el HTML descargado en texto.

Primero elimino elementos que normalmente no forman parte del contenido textual
principal, como scripts, estilos y código embebido.

Luego utilizo `get_text()` para extraer el texto visible.

Finalmente realizo una limpieza básica de espacios y líneas vacías.

In [7]:
def extraer_texto(html):

    soup = BeautifulSoup(
        html,
        "lxml"
    )

    for elemento in soup([
        "script",
        "style",
        "noscript",
        "iframe",
        "svg"
    ]):
        elemento.decompose()

    texto = soup.get_text(
        separator="\n"
    )

    lineas = []

    for linea in texto.splitlines():

        linea_limpia = " ".join(
            linea.split()
        )

        if linea_limpia:
            lineas.append(
                linea_limpia
            )

    return "\n".join(lineas)

## Prueba de extracción de texto

En esta celda aplico la función de extracción sobre el HTML descargado.

De esta manera puedo verificar la diferencia entre el contenido HTML original y
el texto que posteriormente utilizaré como documento para el sistema RAG.

In [8]:
texto_prueba = extraer_texto(
    html_prueba
)

print(texto_prueba[:5000])

Inicio - Ayuda sobre el monotributo - Monotributo | ARCA
Evitar las herramientas de navegaciÃ³n y pasar al contenido
Monotributo
Menu
Inicio
Ayuda
Inicio
Ayuda sobre el monotributo
Inicio
Ayuda sobre el monotributo
Toda la informaciÃ³n sobre cÃ³mo darte de alta y hacer operaciones como monotributista.
Ingresar con clave fiscal
MenÃº de contenidos
QuÃ© es
INSCRIPCIÃN
Inicio
Clave fiscal
CUIT
Domicilio Fiscal ElectrÃ³nico
Jurisdicciones
Actividades
ALTA DE MONOTRIBUTO
Procedimiento
Tipos de monotributo
ParÃ¡metros
JubilaciÃ³n
Obra social
Monotributo unificado
Constancias y credenciales
DESPUÃS DEL ALTA
Desarrollo de la actividad
FacturaciÃ³n
Pagos
RecategorizaciÃ³n
FINALIZACIÃN DE ACTIVIDADES
Baja
Por cese de actividades
De oficio
ExclusiÃ³n
Renuncia
Pasaje al rÃ©gimen general
Ayuda
Inicio
El primer paso es inscribirse ante ARCA para poder despuÃ©s darse de alta en impuestos y utilizar los servicios con clave fiscal.
Para ello es necesario obtener la
clave fiscal
y la
CUIT
.
Luego, ha

## Descarga y procesamiento de todas las fuentes

En esta celda recorro todas las fuentes definidas previamente.

Para cada fuente realizo los siguientes pasos:

1. Descargo la página.
2. Extraigo su texto.
3. Creo un documento estructurado.
4. Guardo información sobre el origen.
5. Registro la fecha de descarga.

También utilizo un bloque `try/except` para evitar que un error en una fuente
interrumpa todo el proceso.

Cada documento queda representado mediante un diccionario con sus metadatos y
su contenido textual.

In [9]:
corpus = []

for fuente in FUENTES_ARCA:

    print("=" * 80)
    print(f"Procesando: {fuente['titulo']}")
    print(f"URL: {fuente['url']}")

    try:

        html = descargar_pagina(
            fuente["url"]
        )

        texto = extraer_texto(
            html
        )

        documento = {
            "id": fuente["id"],
            "titulo": fuente["titulo"],
            "organismo": "ARCA",
            "url": fuente["url"],
            "fecha_descarga": datetime.now(
                timezone.utc
            ).isoformat(),
            "texto": texto
        }

        corpus.append(
            documento
        )

        print(
            f"OK - {len(texto)} caracteres extraídos"
        )

    except Exception as error:

        print(
            f"ERROR: {error}"
        )

print("\n" + "=" * 80)
print("PROCESO TERMINADO")
print("=" * 80)
print(
    f"Documentos descargados correctamente: {len(corpus)}"
)

Procesando: Inicio - Ayuda sobre el Monotributo
URL: https://www.arca.gob.ar/monotributo/ayuda/inicio.asp
OK - 1994 caracteres extraídos
Procesando: Obtención de Clave Fiscal
URL: https://www.arca.gob.ar/monotributo/ayuda/clave-fiscal.asp
OK - 2666 caracteres extraídos
Procesando: Constancias y credenciales
URL: https://www.arca.gob.ar/monotributo/ayuda/constancias-y-credenciales.asp
OK - 2626 caracteres extraídos
Procesando: Facturación
URL: https://www.arca.gob.ar/monotributo/ayuda/facturacion.asp
OK - 4305 caracteres extraídos
Procesando: Recategorización
URL: https://ftp.arca.gob.ar/monotributo/ayuda/recategorizacion.asp
OK - 4704 caracteres extraídos
Procesando: Baja de monotributo
URL: https://www.arca.gob.ar/monotributo/ayuda/baja.asp
OK - 2301 caracteres extraídos
Procesando: Desarrollo de la actividad
URL: https://www.arca.gob.ar/monotributo/ayuda/desarrollo-de-la-actividad.asp
OK - 2203 caracteres extraídos
Procesando: Tutoriales sobre Monotributo
URL: https://www.arca.gob.ar

## Resumen del corpus obtenido

En esta celda reviso los documentos descargados.

Para cada documento muestro su identificador, título y cantidad de caracteres.

Esto permite detectar rápidamente si alguna fuente fue descargada con una
cantidad anormalmente pequeña de texto.

In [10]:
print("RESUMEN DEL CORPUS")
print("=" * 80)

for documento in corpus:

    print(f"\nID: {documento['id']}")
    print(f"Título: {documento['titulo']}")
    print(
        f"Caracteres: {len(documento['texto'])}"
    )
    print(f"URL: {documento['url']}")

RESUMEN DEL CORPUS

ID: inicio
Título: Inicio - Ayuda sobre el Monotributo
Caracteres: 1994
URL: https://www.arca.gob.ar/monotributo/ayuda/inicio.asp

ID: clave_fiscal
Título: Obtención de Clave Fiscal
Caracteres: 2666
URL: https://www.arca.gob.ar/monotributo/ayuda/clave-fiscal.asp

ID: constancias
Título: Constancias y credenciales
Caracteres: 2626
URL: https://www.arca.gob.ar/monotributo/ayuda/constancias-y-credenciales.asp

ID: facturacion
Título: Facturación
Caracteres: 4305
URL: https://www.arca.gob.ar/monotributo/ayuda/facturacion.asp

ID: recategorizacion
Título: Recategorización
Caracteres: 4704
URL: https://ftp.arca.gob.ar/monotributo/ayuda/recategorizacion.asp

ID: baja
Título: Baja de monotributo
Caracteres: 2301
URL: https://www.arca.gob.ar/monotributo/ayuda/baja.asp

ID: desarrollo_actividad
Título: Desarrollo de la actividad
Caracteres: 2203
URL: https://www.arca.gob.ar/monotributo/ayuda/desarrollo-de-la-actividad.asp

ID: tutoriales
Título: Tutoriales sobre Monotributo
C

## Visualización de un documento

En esta celda inspecciono manualmente uno de los documentos del corpus.

Esta revisión es importante porque permite verificar la calidad de la extracción
antes de continuar con las siguientes etapas del proyecto.

En particular, quiero comprobar que el texto contiene información útil y que no
estoy almacenando solamente elementos de navegación o contenido irrelevante.

In [11]:
documento = corpus[0]

print("=" * 80)
print(documento["titulo"])
print("=" * 80)

print(documento["texto"][:10000])

Inicio - Ayuda sobre el Monotributo
Inicio - Ayuda sobre el monotributo - Monotributo | ARCA
Evitar las herramientas de navegaciÃ³n y pasar al contenido
Monotributo
Menu
Inicio
Ayuda
Inicio
Ayuda sobre el monotributo
Inicio
Ayuda sobre el monotributo
Toda la informaciÃ³n sobre cÃ³mo darte de alta y hacer operaciones como monotributista.
Ingresar con clave fiscal
MenÃº de contenidos
QuÃ© es
INSCRIPCIÃN
Inicio
Clave fiscal
CUIT
Domicilio Fiscal ElectrÃ³nico
Jurisdicciones
Actividades
ALTA DE MONOTRIBUTO
Procedimiento
Tipos de monotributo
ParÃ¡metros
JubilaciÃ³n
Obra social
Monotributo unificado
Constancias y credenciales
DESPUÃS DEL ALTA
Desarrollo de la actividad
FacturaciÃ³n
Pagos
RecategorizaciÃ³n
FINALIZACIÃN DE ACTIVIDADES
Baja
Por cese de actividades
De oficio
ExclusiÃ³n
Renuncia
Pasaje al rÃ©gimen general
Ayuda
Inicio
El primer paso es inscribirse ante ARCA para poder despuÃ©s darse de alta en impuestos y utilizar los servicios con clave fiscal.
Para ello es necesario obtener l

## Guardado del corpus en formato JSON

En esta celda guardo todo el corpus en un archivo JSON.

El archivo contiene tanto el texto de cada documento como sus metadatos.

Esto me permite separar la etapa de adquisición de datos de las etapas
posteriores. En los próximos notebooks podré cargar directamente este corpus
sin tener que descargar nuevamente todas las páginas.

El archivo será la entrada para la etapa de chunking y generación de embeddings.

In [12]:
NOMBRE_ARCHIVO = "corpus_arca_monotributo.json"

with open(
    NOMBRE_ARCHIVO,
    "w",
    encoding="utf-8"
) as archivo:

    json.dump(
        corpus,
        archivo,
        ensure_ascii=False,
        indent=4
    )

print(
    f"Corpus guardado en: {NOMBRE_ARCHIVO}"
)

Corpus guardado en: corpus_arca_monotributo.json


## Verificación del archivo guardado

En esta celda vuelvo a cargar el archivo JSON que acabo de generar.

Esta verificación permite comprobar que el archivo fue guardado correctamente y
que contiene la misma cantidad de documentos que el corpus utilizado en memoria.

In [13]:
with open(
    NOMBRE_ARCHIVO,
    "r",
    encoding="utf-8"
) as archivo:

    corpus_verificado = json.load(
        archivo
    )

print(
    f"Documentos cargados desde JSON: "
    f"{len(corpus_verificado)}"
)

print("\nDocumentos:")

for documento in corpus_verificado:
    print(
        f"- {documento['titulo']}"
    )

Documentos cargados desde JSON: 8

Documentos:
- Inicio - Ayuda sobre el Monotributo
- Obtención de Clave Fiscal
- Constancias y credenciales
- Facturación
- Recategorización
- Baja de monotributo
- Desarrollo de la actividad
- Tutoriales sobre Monotributo


## Conclusión del notebook

En este notebook construí el corpus documental inicial para el proyecto.

El corpus contiene información extraída automáticamente desde páginas oficiales
de ARCA relacionadas con trámites de Monotributo.

Cada documento conserva:

- Un identificador.
- El título de la fuente.
- El organismo de origen.
- La URL oficial.
- La fecha de descarga.
- El texto extraído.

En el siguiente notebook voy a cargar este corpus y dividir cada documento en
fragmentos más pequeños mediante una estrategia de chunking.

Posteriormente generaré embeddings para esos fragmentos y los almacenaré en una
base vectorial.